# ERA 5 NetCDF File Division

by Junqi Qiu
assisted by Google Gemini

In this notebook, NetCDF files with multiple variables are divided into several files with only one variable.

In [1]:
# paths

path_raw_1='/home/jqiu/analysis-junqi/Temp_ERA5_Division/Data_raw/888db66dafbc9adc68b5f71df4129dfb.nc'

path_raw_2='/home/jqiu/analysis-junqi/Temp_ERA5_Division/Data_raw/data_stream-oper_stepType-accum.nc'

path_raw_3='/home/jqiu/analysis-junqi/Temp_ERA5_Division/Data_raw/data_stream-oper_stepType-instant.nc'

path_output='/home/jqiu/analysis-junqi/Temp_ERA5_Division/Data_processed'

In [4]:
import xarray as xr

ds_1=xr.load_dataset(path_raw_1)

print(ds_1)

<xarray.Dataset> Size: 51MB
Dimensions:         (valid_time: 1464, pressure_level: 1, latitude: 93,
                     longitude: 93)
Coordinates:
    number          int64 8B 0
  * valid_time      (valid_time) datetime64[ns] 12kB 2022-06-01 ... 2022-07-3...
  * pressure_level  (pressure_level) float64 8B 1e+03
  * latitude        (latitude) float64 744B 25.0 24.75 24.5 ... 2.5 2.25 2.0
  * longitude       (longitude) float64 744B 103.0 103.2 103.5 ... 125.8 126.0
    expver          (valid_time) <U4 23kB '0001' '0001' '0001' ... '0001' '0001'
Data variables:
    q               (valid_time, pressure_level, latitude, longitude) float32 51MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-06T22:34 GRIB to CDM+CF via cfgrib-

In [6]:
# processing

import xarray as xr
import pandas as pd
import os


raw_files = [path_raw_1, path_raw_2, path_raw_3]

# 确保输出目录存在
os.makedirs(path_output, exist_ok=True)

# 核心拆分函数 (按变量 + 按月) 
def split_nc_by_var_and_month(input_path, output_dir):
    print(f"\n正在处理文件: {os.path.basename(input_path)}")
    
    with xr.open_dataset(input_path) as ds:
        variables = list(ds.data_vars)
        print(f"发现 {len(variables)} 个变量: {variables}")
        
        # 检查是否包含 valid_time 维度
        if 'valid_time' not in ds.dims:
            print(f"  -> [跳过] 该文件没有 valid_time 维度，无法按月拆分。")
            return

        # 智能提取文件中包含的所有唯一“年-月”组合 (例如: '2022-06', '2022-07')
        times = pd.DatetimeIndex(ds.valid_time.values)
        year_months = times.to_period('M').unique()
        print(f"发现数据包含以下月份: {[str(ym) for ym in year_months]}")
        
        # 第一层循环：遍历所有变量
        for var in variables:
            ds_single_var = ds[[var]]
            
            # 第二层循环：遍历所有月份
            for ym in year_months:
                month_str = str(ym)  # 将周期格式化为字符串，如 '2022-06'
                
                # 利用 xarray 强大的按时间字符串切片的功能，直接切出这一个月的数据
                ds_month = ds_single_var.sel(valid_time=month_str)
                
                # 构造新的文件名，格式：变量名_时间.nc (例如: u10_2022-06.nc)
                out_filename = f"{var}_{month_str}.nc"
                out_path = os.path.join(output_dir, out_filename)
                
                # 保存为独立的 nc 文件
                ds_month.to_netcdf(out_path)
                print(f"  -> 已导出: {out_filename}")

# 批量执行
for file_path in raw_files:
    if os.path.exists(file_path):
        split_nc_by_var_and_month(file_path, path_output)
    else:
        print(f"\n[警告] 找不到文件，请检查路径: {file_path}")

print("\n 所有文件已按【变量+月份】拆分完成！")


正在处理文件: 888db66dafbc9adc68b5f71df4129dfb.nc
发现 1 个变量: ['q']
发现数据包含以下月份: ['2022-06', '2022-07']
  -> 已导出: q_2022-06.nc
  -> 已导出: q_2022-07.nc

正在处理文件: data_stream-oper_stepType-accum.nc
发现 1 个变量: ['tp']
发现数据包含以下月份: ['2022-06', '2022-07']
  -> 已导出: tp_2022-06.nc
  -> 已导出: tp_2022-07.nc

正在处理文件: data_stream-oper_stepType-instant.nc
发现 5 个变量: ['u10', 'v10', 't2m', 'sst', 'lsm']
发现数据包含以下月份: ['2022-06', '2022-07']
  -> 已导出: u10_2022-06.nc
  -> 已导出: u10_2022-07.nc
  -> 已导出: v10_2022-06.nc
  -> 已导出: v10_2022-07.nc
  -> 已导出: t2m_2022-06.nc
  -> 已导出: t2m_2022-07.nc
  -> 已导出: sst_2022-06.nc
  -> 已导出: sst_2022-07.nc
  -> 已导出: lsm_2022-06.nc
  -> 已导出: lsm_2022-07.nc

 所有文件已按【变量+月份】拆分完成！
